# LSTMs: An introduction to the structure

Welcome to sequence modeling. Unlike standard feedforward neural networks that process individual, isolated data points (like a single image), LSTMs process **Sequential Data** (like a sentence, a stock price over time, or an audio wave).

### The Mathematical Core: The Gates
An LSTM cell contains four main interacting neural network layers:
1. **Forget Gate ($f_t$):** Decides what information to throw away from the long-term cell state.
   $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
2. **Input Gate ($i_t$) & Candidate Memory ($\tilde{C}_t$):** Decides what *new* information to store in the cell state.
   $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
   $$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
3. **Cell State Update ($C_t$):** The actual mathematical update of the long-term memory.
   $$C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$$
4. **Output Gate ($o_t$):** Decides what parts of the cell state to output as the new hidden state ($h_t$).
   $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
   $$h_t = o_t * \tanh(C_t)$$

### Our Analytical Pipeline:
1. **Environment Setup:** Importing PyTorch.
2. **Sequential Data Generation:** Synthesizing a temporal sine wave dataset.
3. **Architecture Design:** Building the LSTM class and analyzing its tensor dimensions.
4. **Training:** Backpropagation through time (BPTT).
5. **Evaluation:** Predicting future sequence values.

In [ ]:
# Cell 2: Environment Setup and Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for analytical reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")

## Step 1: Generating Sequential Data

To train an LSTM, we need a sequence where the past dictates the future. We will generate a simple **Sine Wave**. 

To feed this into an LSTM, we must use a **Sliding Window**. If our window size is 10, the input ($X$) will be the sequence of points `[t_1, t_2 ... t_10]`, and the target ($y$) will be the very next point `t_11`.

In [ ]:
# Cell 4: Data Generation and Windowing

# 1. Generate a noisy sine wave
time_steps = np.linspace(0, 100, 1000)
sine_wave = np.sin(time_steps) + np.random.normal(0, 0.05, 1000) # Added slight noise

# 2. Define the sliding window function
def create_sequences(data, window_size):
    sequences = []
    targets = []
    for i in range(len(data) - window_size):
        seq = data[i:i + window_size]
        label = data[i + window_size]
        sequences.append(seq)
        targets.append(label)
    return np.array(sequences), np.array(targets)

window_size = 20
X, y = create_sequences(sine_wave, window_size)

# 3. Convert to PyTorch Tensors
# LSTMs expect 3D tensors: (Batch Size, Sequence Length, Input Size)
# Our input size is 1 (a single number at each time step)
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1) 
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

# Split into Training (80%) and Testing (20%)
split_idx = int(len(X) * 0.8)
X_train, X_test = X_tensor[:split_idx], X_tensor[split_idx:]
y_train, y_test = y_tensor[:split_idx], y_tensor[split_idx:]

print("--- Analytical Tensor Shapes ---")
print(f"Training Input Shape:  {X_train.shape} -> (Batch: {X_train.shape[0]}, Seq_Len: {X_train.shape[1]}, Input_Dim: {X_train.shape[2]})")
print(f"Training Target Shape: {y_train.shape} -> (Batch: {y_train.shape[0]}, Output_Dim: {y_train.shape[1]})")

## Step 2: Building the LSTM Architecture

In PyTorch, we inherit from `nn.Module`. 

We will use PyTorch's highly optimized `nn.LSTM` layer. However, the LSTM layer outputs a high-dimensional hidden state ($h_t$) for every step in the sequence. Because we only want to predict a single continuous number (the next point in the wave), we must pass the *final* hidden state of the sequence into a standard `nn.Linear` (Dense) layer to shrink the dimension down to 1.

In [ ]:
# Cell 6: Defining the LSTM Model

class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(TimeSeriesLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        
        # 1. The Core LSTM Layer
        # batch_first=True tells PyTorch our tensors are (Batch, Seq, Feature)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # 2. The Final Output Layer
        # Maps the complex hidden state back down to a single numeric prediction
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # The LSTM returns two things:
        # out: The hidden states for EVERY time step (Batch, Seq, Hidden_Size)
        # (h_n, c_n): The final hidden and cell states for the sequence
        out, (h_n, c_n) = self.lstm(x)
        
        # Analytically, we only care about the very last hidden state to make our prediction
        # We slice the tensor to get the last time step: out[:, -1, :]
        final_hidden_state = out[:, -1, :]
        
        # Pass the final hidden state through the linear layer
        prediction = self.linear(final_hidden_state)
        return prediction

# Instantiate the model
model = TimeSeriesLSTM()
print(model)

## Step 3: Loss, Optimizer, and Training Loop

Because we are predicting a continuous continuous value (Regression), we use **Mean Squared Error (MSE)** as our Loss Function. We use the **Adam** optimizer to perform Gradient Descent.

Unlike standard networks, training an LSTM involves **Backpropagation Through Time (BPTT)**. PyTorch automatically unrolls the temporal sequence in its computational graph to calculate how much weight to assign to an error made 20 time steps ago!

In [ ]:
# Cell 8: The Training Loop

# Hyperparameters
epochs = 150
learning_rate = 0.01

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("--- Starting Training ---")
loss_history = []

for epoch in range(epochs):
    # 1. Forward Pass
    predictions = model(X_train)
    
    # 2. Calculate Mathematical Error
    loss = criterion(predictions, y_train)
    
    # 3. Backpropagation Through Time
    optimizer.zero_grad() # Clear old gradients
    loss.backward()       # Calculate new gradients
    optimizer.step()      # Update the weights
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")

print("Training Complete!")

## Step 4: Analytical Evaluation and Visualization

To prove the LSTM learned the temporal structure, we will feed it our completely unseen `X_test` data. 

We will overlay the model's predictions on top of the actual true sine wave to visually analyze its predictive accuracy and ensure it captured the amplitude and frequency correctly.

In [ ]:
# Cell 10: Evaluation and Plotting

# Turn off gradient tracking for evaluation to save memory and compute
model.eval()
with torch.no_grad():
    # Make predictions on the test set
    test_predictions = model(X_test).numpy()
    
# Calculate final test error
test_loss = criterion(torch.tensor(test_predictions), y_test).item()
print(f"Final Test MSE Loss: {test_loss:.6f}\n")

# Realign the arrays for plotting so they match up chronologically
# The test set starts at index `split_idx + window_size` of the original sine wave
time_test = time_steps[split_idx + window_size:]
true_values = y_test.numpy()

# Visualizing the Results
plt.figure(figsize=(12, 5))
plt.plot(time_test, true_values, label="True Ground Truth", color="blue", linewidth=2, alpha=0.6)
plt.plot(time_test, test_predictions, label="LSTM Predictions", color="red", linestyle="--", linewidth=2)

plt.title("LSTM Temporal Prediction Analysis on Unseen Data", fontsize=15)
plt.xlabel("Time Steps", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

print("Analytical Conclusion: The LSTM successfully utilized its internal Cell State to remember the cyclical frequency of the wave, perfectly overlapping the true trajectory even amidst the injected noise!")